# Train yolo-pill-classifier on Google Colab

`dataset/`(960x960 리사이즈 완료, 820장)이 GitHub 저장소에 함께 올라가 있어
**clone만 하면 코드와 데이터가 같이 받아집니다.** Drive 업로드는 필요 없습니다.

**실행 전:** `런타임 > 런타임 유형 변경 > T4 GPU` 선택.

**주의:** 로컬에서 코드를 고쳤다면 먼저 `git push` 해야 여기에 반영됩니다.

In [ ]:
# 0. GPU 확인 (T4가 안 뜨면 런타임 유형부터 바꾸세요)
!nvidia-smi

In [ ]:
# 1. 코드 + 데이터셋 + 배포 모델 clone (학습 중간 산출물 runs/는 제외)
!git clone https://github.com/clanadian/yolo-pill-classifier.git
%cd yolo-pill-classifier

In [ ]:
# 2. 의존성 설치
!pip install -q -r requirements.txt

In [ ]:
# 3. 데이터셋 확인 (train 919 / val 161이 나와야 정상)
!echo train: $(find dataset/images/train -type f | wc -l)
!echo val:   $(find dataset/images/val   -type f | wc -l)

In [ ]:
# 4. 학습 (--workers 2: Colab 런타임은 CPU 2코어라 기본 8은 과함)
#    스크립트가 시작할 때 라벨 짝/클래스 분포를 먼저 검사하고,
#    data.yaml이 없으면 Colab 경로 기준으로 새로 만들어 줍니다.
!python training/train_yolo.py --device 0 --workers 2 --export-best

In [ ]:
# 5. best.pt를 Drive에 백업 (런타임 초기화돼도 유실 안 되게)
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/yolo-pill-classifier', exist_ok=True)
shutil.copy2('model/best.pt', '/content/drive/MyDrive/yolo-pill-classifier/best.pt')
print('Saved to Drive.')

In [ ]:
# 6. 결과 확인 (confusion matrix로 색 비슷한 알약이 섞이는지 보기)
from IPython.display import Image, display
import glob
d = sorted(glob.glob('runs/pill_yolo*'))[-1]
print(d)
for f in ('confusion_matrix_normalized.png','results.png'):
    display(Image(filename=f'{d}/{f}', width=760))